## Tests — Regression Prediction (Runway / Overspend Risk)

Validates the regression feature builder and training utilities in `model/predict_core.py`.

These tests do not read the real dataset; they use small synthetic daily spend to keep tests fast and deterministic.

## Imports

In [2]:
from __future__ import annotations

import sys
from datetime import date
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing a 'data/' directory")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.predict_core import (
    build_monthly_training_samples,
    derive_overspend_risk_and_days_to_limit,
    parse_currency_to_float,
    train_month_end_spend_model,
)

## Synthetic daily discretionary spend

In [3]:
days = pd.date_range("2010-01-01", "2010-02-20", freq="D")

# Client 1: steady spend with a small ramp
c1 = pd.DataFrame(
    {
        "client_id": 1,
        "day": days,
        "discretionary_spend_usd": [float(2 + (i % 5)) for i in range(len(days))],
    }
)

# Client 2: higher spend with occasional refunds (negative values should not increase spend)
c2 = pd.DataFrame(
    {
        "client_id": 2,
        "day": days,
        "discretionary_spend_usd": [float(5 + (i % 7)) for i in range(len(days))],
    }
)
c2.loc[c2.index[::13], "discretionary_spend_usd"] = -10.0

daily = pd.concat([c1, c2], ignore_index=True)

user_limits = pd.Series({1: 120.0, 2: 240.0})


## Build training samples (features + month-end target)

In [4]:
samples = build_monthly_training_samples(daily, user_limits=user_limits)
assert not samples.empty
required = {
    "client_id",
    "day",
    "ym",
    "month_total_discretionary_spend_usd",
    "mtd_discretionary_spend_usd",
    "monthly_discretionary_limit_usd",
    "avg_daily_discretionary_spend_7d",
    "avg_daily_discretionary_spend_30d",
}
assert required.issubset(set(samples.columns))
assert samples["month_total_discretionary_spend_usd"].ge(0).all()
assert samples["mtd_discretionary_spend_usd"].ge(0).all()


## Train regression model (time-based split)

In [5]:
artifacts = train_month_end_spend_model(samples, cutoff_day=date(2010, 1, 25))
assert "mae_test_usd" in artifacts.metrics
assert artifacts.metrics["n_train"] >= 10
assert artifacts.metrics["n_test"] >= 10


## Derive overspending risk + days-to-limit estimate

In [6]:
row = samples.iloc[0]
feature_row = {k: row[k] for k in artifacts.feature_columns}
pred = float(artifacts.model.predict(pd.DataFrame([feature_row]))[0])

signals = derive_overspend_risk_and_days_to_limit(
    predicted_month_end_discretionary_spend_usd=pred,
    mtd_discretionary_spend_usd=float(row["mtd_discretionary_spend_usd"]),
    monthly_discretionary_limit_usd=float(row["monthly_discretionary_limit_usd"]),
    as_of_day=pd.to_datetime(row["day"]).date(),
    days_in_month=int(row["days_in_month"]),
)
assert set(signals.keys()) == {
    "overspend_risk",
    "days_to_limit_estimate",
    "remaining_budget_usd",
}

# If already over budget, estimate should be 0
signals2 = derive_overspend_risk_and_days_to_limit(
    predicted_month_end_discretionary_spend_usd=999.0,
    mtd_discretionary_spend_usd=300.0,
    monthly_discretionary_limit_usd=200.0,
    as_of_day=date(2010, 2, 10),
    days_in_month=28,
)
assert signals2["days_to_limit_estimate"] == 0


## Currency parsing sanity checks

In [7]:
assert parse_currency_to_float("$2,238 ") == 2238.0
assert parse_currency_to_float(None) is None
assert parse_currency_to_float(" ") is None

print("Predict tests: PASS")


Predict tests: PASS
